(Per l'indice, vedi Outline in basso a sx).

### Classification Fundamentals
Classification falls under supervised learning, i.e. the model is trained on a labeled dataset. The goal of classification is to predict the categorical (so, discrete) class labels of new instances based on the patterns learned from the training data.

We start from a **training dataset** which contains instances with known class labels. Starting from this dataset, we train a classification model that learns to associate input features with their corresponding class labels. Once trained, the model can be used to predict the class labels of new, unseen instances.
The output of a classification model is usually a class label (e.g., "spam" or "not spam" in email classification) or a probability distribution over multiple classes (e.g., the likelihood of an image belonging to different categories like "cat," "dog," or "bird", if we want a single affermation we get the one with most probability).

The **test set** is a separate dataset that is used to evaluate the performance of the trained classification model. It contains instances with known class labels, but these instances were not used during the training phase. The purpose of the test set is to assess how well the model generalizes to new, unseen data.

There are lots of classification techniques. For each of those, we will look a bunch of pros and cons, in terms of:
- **Accuracy**: How well does the model perform on the training data and on unseen data? (Quality of predictions)
- **Interpretability**: How easy is it to understand how the model makes its predictions? (Model Interpretability and Compactness). It is important in domains where understanding the decision-making process is crucial, such as healthcare or finance.
- **Incrementality**: Can the model be updated with new data without retraining from scratch? (Model Update Efficiency)
- **Scalability**: How well does the model handle large datasets and high-dimensional feature spaces? 
- **Efficiency**: How computationally efficient is the model in terms of training time and prediction time? (Computational Efficiency)
- **Robustness**: How well does the model handle noisy or incomplete data? 

### Hunt's Algorithm
For the example, we will see the following toy dataset:

<img src="img_teoria/Dataset_tree.png" width="50%">

Each node of the tree represents a "Spitting Attribute", each branch represents a decision rule, and each leaf node represents a class label (the decision taken after computing all attributes). The topmost node in a decision tree is the root node.

Given the tree, we can classify a new instance by traversing the tree from the root to a leaf node, following the decision rules based on the attribute values of the instance and arriving at a class label.

Our goal is to study Hunt's Algorithm, a greedy algorithm that creates a reasonable decision tree from the training data.

**General structure of the algorithm:**
The algorithm is recursive, splitting the dataset into smaller subsets by selecting an attribute to split each time. So, starting from the whole dataset Dt:
1. If Dt contains records that belong to more than one class:
    - select the "best" attribute A to split Dt (we will see how to do this later) and label node t as A
    - split Dt into smaller subsets Dt1, Dt2, ..., Dtn based on the values of attribute A and apply the algorithm recursively to each subset
2. If Dt contains records that all belong to the same class Y_t:
    - label node t as a leaf node with class Y_t, it's a leaf node (we found a "pure" partition)
3. If Dt is empty:
    - label node t as a leaf node with the class Y_d, which is the majority class of the parent node's dataset. (ex. sepal length <= 5 in one node and sepal length > 5 in the other, could lead to an empty dataset for the second node if there are no instances that satisfy sepal length > 5)

Let's see an example of how the algorithm works.


The dumbest tree that we could create is the one that classifies everything as "No" (Don't Cheat), since the majority of the instances are "No".
If the best attribute to split on is "Refund" and we split only in it, then we would get leaves of Don't Cheat since for Refund = No, the majority is No, and for Refund = Yes, the majority is also No.
To create a better tree, we see that for Yes all where No -> pure partition, but for Refund = No we have both Yes and No, so we need to split again. Imagine now the best attribute to split on is "Marital Status", we repeat the same reasoning as before, until we reach pure partitions or empty datasets.

<img src="img_teoria/Hunt's_example.png" width="50%">


This algorithm adopts a greedy strategy, because at each step it selects the attribute that seems to be the best choice at that moment, without considering the overall structure of the tree. This means that while it may not always produce the optimal decision (global optimum) tree, it often results in a reasonably good tree in a relatively short amount of time.

Still open questions to answer:
- How to structure the test condition? (binary split vs multiway split)
- How to choose the best attribute?
- When to stop splitting? (if we get all leaves pure, we could have overfitting problems)

### How to structure the test condition
This depends on the type of attribute we are using to split the dataset **(nominal, ordinal, or continuous)** and on the number of outgoing edges we want from each node **(binary split vs multiway split)**.

- Nominal attributes: These are categorical attributes with no inherent order (e.g., color, type). For nominal attributes, we can create a multiway split where each branch corresponds to a different category of the attribute. For example, if we have an attribute "Color" with values {Red, Blue, Green}, we can create three branches from the node, one for each color.
The other option is to create a binary split, where we select one category and create two branches: one for that category and another for all other categories combined. For example, we can create a binary split for "Color" with branches "Red" and "Not Red".

![](img_teoria/2_nomina.png) ![](img_teoria/1_nomina.png) 
- Ordinal attributes: These are categorical attributes with a specific order (e.g., low, medium, high). For ordinal attributes, we can create a binary split based on a threshold value. For example, if we have an attribute "Satisfaction" with values {Low, Medium, High}, we can create a binary split with branches "Satisfaction <= Medium" and "Satisfaction > Medium". Alternatively, we can create a multiway split with branches for each category.

Poiché gestire direttamente stringhe o categorie multiple può essere complesso, oggi si preferisce convertire gli attributi nominali in una serie di attributi binari (one-hot encoding) per semplificare il processo di split. Ex. we have an attribute "Color" with values {Red, Blue, Green}, we can create three binary attributes: "Is_Red", "Is_Blue", and "Is_Green". If Is_Red = True for a feature, then we follow the branch Color = Red etc...

For Ordinal attributes, we can encode them with integer values that reflect their order. Ex. we can encode "Low" as 1, "Medium" as 2, and "High" as 3. This way, splits only consider numerical thresholds (ex. Satisfaction <= 2 == Low or Medium).

- Continuous attributes: These are numerical attributes that can take any value within a range (e.g., age, income). To handle them, a first technique was to discretize them to form an ordinal categorical attribute by creating intervals (bins). For example, we can discretize the "Age" attribute into bins like {0-18, 19-35, 36-50, 51+}. Then, we can create a multiway split with branches for each bin or a binary split based on a threshold value (e.g., Age <= 35 and Age > 35).
However, a more common approach today is to create binary splits based on threshold values. For example, we can create a binary split for the "Age" attribute with branches "Age <= 30" and "Age > 30". The threshold value can be determined based on the distribution of the data or by using techniques like information gain or Gini index to find the optimal split point, as we will see later.

### How to choose the best attribute
To choose the best attribute to split the dataset at each node, we can use various criteria that measure the quality of the split. The goal is to select the attribute that maximizes the separation of classes in the resulting subsets. In the following example, the best attribute is clearly the second since it creates the purest partitions.

![](img_teoria/best_attr.png)

One way to measure the quality of a split is to use impurity measures, which quantify how mixed the classes are in a subset. Common impurity measures include:
- Gini Index
- Entropy
- Information Gain

The more pure the partitions, the better the split, the lower the impurity, the lower the values of these measures. Having better splits means having better classification performance, since we reach pure partitions (leaves) faster.
Different algorithms rely on different impurity measures to select the best attribute for splitting.
#### Gini Index
Let's treat Gini Index as something that is large in case of impure partitions and small in case of purer partitions. Imagine that we have two attributes A and B, and we want to choose the best one to split our dataset. For each partition, we can compute the Gini Index M1, M2, M3 and M4 and mix them in M12 and M34 for A and B respectively (ossia se scelgo A allora il dataset è splittato in due nodi, il primo avrà gini index M1 e il secondo M2, e la media pesata sarà M12. Scegliendo B invece avremo M3 e M4 e la media pesata M34). Imagine also to have M0 as the Gini Index of the whole dataset before splitting. We can compute the Gini Index "Gain" for both attributes as follows: <br>
<br>
$Gini Gain(A) = M0 - M12$  
$Gini Gain(B) = M0 - M34$  
<br>
The attribute with the highest Gini Gain (so, with the lowest Gini Index overall) is selected as the best attribute to split the dataset.

The Gini Index for a given node t is calculated as follows: <br>
<br>
$GINI(t) = 1 - \sum_{j=1}^{c} p(j|t)^2$  
<br>
where p(j|t) is the proportion of instances in node t that belong to class j (number of instances of class j divided by total instances in node t), and c is the total number of classes.  
The maximum value of GINI(t) is 1-1/c, which occurs when the instances are evenly distributed among all classes (maximum impurity). The minimum value is 0, which occurs when all instances belong to a single class (pure partition).  

**NB.** *c is the total number of classes that we want to classify*

<img src="img_teoria/gini_example.png" width="50%">
<img src="img_teoria/gini_calc.png" width="50%">

This way we can comput M1, M2, M3 and M4. For computing M12 and M34 we use the following formula. When a node p is split into k partitions (children), the quality of the split is computed as: <br>
<br>
$GINI_{split}(A) = \sum_{i=1}^{k} \frac{n_i}{n} GINI(i)$
<br> <br>
where k is the number of partitions created by the split, n_i is the number of records/instances in partition i (for child i), n is the total number of instances in the original dataset, and GINI(i) is the Gini Index of partition i.  
This way we are computing a mean of the Gini Indexes of the children, weighted by the number of instances in each child. In fact if the split produced a child with few instances and another with many instances, the Gini Index of the child with many instances will have a greater impact on the overall GINI_split value.

<img src="img_teoria/gini_tot.png" width="50%">

What seen above is the Gini Index computed for a binary split. For multiway splits, the formula remains the same, but k will be greater than 2, and we will compute the Gini Index for each partition created by the split. 

- When we have **categorical attributes (i.e., nominal or ordinal)**, we can either create multiway splits or binary splits, as explained before. If we choose to use a binary split and the attribute has multiple categories, we can evaluate all possible binary partitions of the categories and select the one that maximizes the Gini Gain. In this way, we can effectively handle categorical attributes while still leveraging the advantages of binary decision trees.  
This idea is conceptually similar to continuous attributes, where the algorithm searches for the optimal split point (threshold) that maximizes impurity reduction.  
For ordinal categorical attributes, another common approach is to encode the categories using integer values that respect the natural order and then perform a binary split via an integer threshold.

![](img_teoria/multi_way.png)

- For **continuous attributes**, we can create binary splits based on threshold values. The algorithm evaluates all possible split points (thresholds) for the continuous attribute and selects the one that maximizes the Gini Gain. This involves sorting the unique values of the continuous attribute and considering midpoints between consecutive values as potential thresholds. The optimal threshold is the one that results in the lowest weighted Gini Index for the resulting partitions. <br>
<br>
So the algorithm works like this:
For each continuous attribute A in the dataset:
   - Sort the unique values of A.
   - For each pair of consecutive values $(v_i, v_{i+1})$, compute the midpoint threshold $t = (v_i + v_{i+1}) / 2$.
   - Linearly scan the sorted values, updating the count matrix as we go, and compute the Gini Index for each threshold t.
   - Choose the threshold t that minimizes the Gini Index as the best split point for attribute A.

![](img_teoria/continuous.png)

Another measure for impurity is Entropy, which is used in algorithms like ID3 and C4.5. Entropy measures the uncertainty or randomness in the dataset. The formula for calculating entropy at a node t is: <br>
$Entropy(t) = - \sum_{j=1}^{c} p(j|t) log_2(p(j|t))$  
where p(j|t) is the proportion of instances in node t that belong to class j, and c is the total number of classes.
Here the maximum value of Entropy(t) is log2(c), which occurs when the instances are evenly distributed among all classes (maximum impurity). The minimum value is 0, which occurs when all instances belong to a single class (pure partition) (log(1) = 0). Here also the gain can be computed as the difference between the entropy of the parent node and the weighted average entropy of the child nodes after the split.

The last important impurity measure is Misclassification Error, which we won't cover in detail here. Below a figure that shows how these three measures behave in relation to the class distribution for two classes, all of them reach the maximum when the classes are evenly distributed and the minimum when all instances belong to a single class.

<img src="img_teoria/confronto.png" width="50%">

### Stopping Criteria (Overfitting)
For now, we stop expanding only when the nodes are pure (all instances belong to the same class) or when the dataset is empty. However, this can lead to overfitting, where the model learns the training data too well and performs poorly on unseen data.  
In the figure below we see the Error % when varying the depth of the tree (number of nodes). As we can see, the training error decreases as the depth increases, but the test error starts to increase after a certain point, indicating **overfitting**.

<img src="img_teoria/overfitting.png" width="50%">

Overfitting is due to the following in decision trees: when we let decision trees grow too deep, they can capture noise and outliers in the training data, leading to poor generalization on unseen data. This is because lower nodes "are referring" to a lower number of instances! To prevent overfitting, we can implement stopping criteria during the tree construction process. 
In the figure below the division between dots and plus could be made with a single split, but in order to perfectly classify all training instances, the tree grows deeper and does more splits

<img src="img_teoria/toosplits.png" width="50%">

To avoid overfitting, we have two main strategies:
1. **Pre-pruning (Early Stopping)**: This involves stopping the tree growth before it becomes too complex. In order to do this we can set conditions such as:
   - Maximum depth of the tree: We can limit how deep the tree can grow. Once the maximum depth is reached, we stop splitting further.
   - Minimum impurity decrease: We can set a threshold for the minimum decrease in impurity (e.g., Gini Index or Entropy) required to justify a split. If the decrease is below this threshold, we stop splitting (in fact if the impurity decrease is very small, it means that the split is not really helping to improve the classification).
   - Minimum number of samples per leaf: We can specify a minimum number of instances that must be present in a leaf node. If a split would result in a leaf with fewer instances than this threshold, we do not perform the split.
2. **Post-pruning**: This involves allowing the tree to grow fully and then pruning back branches that do not contribute significantly to the model's performance on a validation set. This can be done by trying different cuts and seeing if the performance of the model improves.

Tipically post-pruning gives better results than pre-pruning, since in pre-pruning we could stop too early and miss important patterns in the data. However, post-pruning can be more computationally expensive since it requires building the full tree first and then pruning it, so it is less commonly used in practice.

### Handling Missing Values and Decision Boundary

Handling missing values in decision trees can be challenging, since:
- it affects the calculation of impurity measures, as missing values can lead to inaccurate estimates of class proportions
- it affects how to distribute instances with missing values to child nodes during splits
- it affects how a test instance with missing values is classified  

-> decision trees are not very robust in managing missing values

The **Decision Boundary** of a decision tree is the set of rules that define how the feature space is divided into different regions, each corresponding to a specific class label. In a decision tree, the decision boundary is formed by the splits at each node, which create hyperplanes that separate the feature space based on the values of the attributes.

Imagine to have 2 classes and 2 features, the decision boundary created by a decision tree would consist of axis-aligned lines (in 2D) that partition the feature space into rectangular regions, each associated with a class label. 

In the image below, the first split creates a vertical line, while the second split and third splits create two horizontal lines, resulting in four distinct regions corresponding to different class labels.  
By doing this, the 2D space is divided into regions where each region corresponds to a specific class label based on the decision rules learned by the tree. So in this case the decision boundary consists of three lines that create four rectangular regions, each associated with a class label based on the splits made by the decision tree. Note that decision boundaries are always orthogonal to the feature axes, this is because splits are made on single attributes at a time.

**More simply, the decision boundary represents the border line that separates different classes in the feature space based on the decisions made by the tree.**

![](img_teoria/decision_boundary.png)

Since the splits are axis-aligned, decision trees can struggle to capture complex decision boundaries that are not aligned with the feature axes. In the example below, the obvious optimal decision boundary is diagonal, but decision trees can only create axis-aligned splits, leading to a deep and complex tree to approximate the diagonal boundary.

![](img_teoria/oblique.png)

A way to avoid this, in this case, is to add a new feature that is a combination of the two original features, x+y, which allows the decision tree to create a split that effectively captures the diagonal decision boundary. So feature engineering (where we can combine multiple features) can help decision trees to better capture complex decision boundaries.

![](img_teoria/x+y.png)

### Recap - Decision Trees

- Accuracy: Decision trees work on only one feature at a time, so they are not the best classifiers in terms of accuracy.
- Interpretability: Decision trees are highly interpretable, as the tree structure provides a clear and intuitive representation of the decision-making process. We can understand what the whole model does and how it makes a specific prediction. (ex. a person cheated because this, this and this...)
- Incrementality: not incremental, if we want to add new data we have to retrain the whole tree from scratch.
- Efficiency: Fast to train (building the model, the tree) and predict, especially for small to medium-sized datasets.
- Scalability: Scalable both in training set size and attribute number. If in fact we have new samples -> we simply must consider more points when we do the splitting. If we have new attributes -> we don't worry about curse of dimensionality since decision trees don't rely on distance metrics. Also if we have more attributes we simply compute more splits, still in scale.
- Robustness: Decision trees are sensitive to noisy data and outliers, which can lead to overfitting. Also missing values must be handled correctly to avoid biasing the splits.

# Random Forests
It's an algorithm still based on decision trees. It is an ensemble learning technique, since it combines multiple models (decision trees) to improve classification performance (accuracy and stability) and avoid overfitting.

Random forest because it's a set of decision trees, random because we need stocasticity to make the trees different from each other and allow the model to work better.  **In order to take a decision, the idea is to take a vote among all the trees and assign the class that gets the most votes.**

The algorithm works as follows:
1. From the original dataset, create multiple random subsets of the data D1, ..., Db by sampling with replacement (bootstrap sampling) -> we could have repeated instances in each subset.
2. While building each decision tree, at each split node, randomly select a subset of features to consider for splitting the data. This introduces additional randomness and diversity among the trees. Tipically, being p the number of features, we select sqrt(p) features at each split. 
3. Train a decision tree on each subset Di, resulting in b different decision trees with different possible predictions, since each tree is trained on a different subset of the data.
4. To classify a new instance, pass it through each of the b decision trees and collect their predictions. The final class label is determined by majority voting, where the class that receives the most votes from the individual trees is assigned to the instance.

2- is useful since for ex. if we have a dataset where we want to predict height and weight is a very important related feature, all trees would use weight to split first, leading to very similar trees. By randomly selecting a subset of features at each split, we "force" some trees to consider different features, leading to more diverse trees and better overall performance.

This way, trees are not strongly correlated, which helps to reduce overfitting and improve the generalization of the model.

### Recap - Random Forests
- Accuracy: Consistently better than individual decision trees, as the ensemble approach helps to reduce overfitting and improve generalization.
- Interpretability: Less interpretable than individual decision trees, as the ensemble of trees makes it harder to understand the overall decision-making process. However, we can use feature importance to estimate which features are most influential in the model's predictions.
- Incrementality: Not incremental, similar to decision trees, adding new data requires retraining the entire forest.
- Efficiency: More computationally intensive than individual decision trees, but very still fast in terms of training and prediction if compared to models. Also parallelizable, since each tree can be trained independently.
- Scalability: Scalable to large datasets and high-dimensional feature spaces, as the ensemble approach allows for efficient training and prediction. (same reasons as decision trees)
- Robustness: Random forests are more robust to noisy data and outliers compared to individual decision trees, as the ensemble approach helps to mitigate the impact of such instances on the overall model performance. (in fact outliers are few by definition, so they will be present only in some trees, not all of them, so their influence is reduced).

# K-Nearest Neighbors (K-NN)
K-NN is a simple yet effective classification algorithm that relies on the concept of similarity between instances. It belongs to the family of **instance-based learning algorithms**, where the model does not explicitly learn a mapping function from input features to class labels during training. Instead, it stores the training instances and uses them directly for making predictions on new instances.

The simplest instance based learning algorithm is Rote-learner, which simply memorizes the training instances and classifies new instances based on the exact match with the stored instances. However, this approach is not very effective for classification tasks, as it does not generalize well to unseen data.

K-NN improves upon this by considering the k-nearest neighbors of a new instance in the feature space.  
K-NN requires:
- The set of stored training instances with their corresponding class labels.
- A distance metric to measure the similarity between instances (e.g., Euclidean distance, Manhattan distance, Minkowski distance, etc.)
- The number of neighbors (k) to consider when making predictions.  

To classify a new instance, K-NN follows these steps:
1. Calculate the distance between the new instance and all stored training instances using the chosen distance metric.
2. Identify the k-nearest neighbors based on the calculated distances.
3. Determine the class label of the new instance by performing a majority vote among the class labels of the k-nearest neighbors. The class that appears most frequently among the neighbors is assigned to the new instance.

Remember that for decision trees, after we build the tree, we don't need the training data anymore, since the model is represented by the tree itself. For K-NN instead, we need to store all the training instances, since they are used directly for making predictions.

**The choice of k is crucial for the performance of the K-NN algorithm**. A small value of k (e.g., k=1) can lead to overfitting, as the model becomes sensitive to noise and outliers in the training data. On the other hand, a large value of k can lead to underfitting, as the model may not capture the local patterns in the data effectively (neighborhood would include points from other classes). So we need to find a balance when selecting k, often through cross-validation or other model selection techniques.

About distances that we usually use in K-NN, **Euclidean Distance** is the most common. It measures the straight-line distance between two points in a multi-dimensional space. The formula for Euclidean distance between two instances x and y with p features is:  

$d(x, y) = \sqrt{\sum_{i=1}^{p} (x_i - y_i)^2}$

A way to improve K-NN performance is to use **Weighted K-NN**, where we assign different weights to the neighbors based on their distance from the new instance. Closer neighbors are given higher weights, while farther neighbors are given lower weights. This way, the influence of each neighbor on the final prediction is proportional to its proximity to the new instance. A common approach is to use the square inverse of the distance as the weight:

$weight_i = \frac{1}{(d(x, x_i))^2}$

Remember that, for K-NN, during the preprocessing phase, it is important to normalize or standardize the feature values to ensure that all features contribute equally to the distance calculations. This is especially important when features have different scales or units (ex. height in cm and weight in kg, sales in $ and age in years).

A problem with distance measures is the **curse of dimensionality**. As the number of features (dimensions) increases, the distance between instances becomes less meaningful, and the performance of K-NN can degrade. This is because, in high-dimensional spaces, instances tend to be equidistant from each other, making it difficult to identify meaningful neighbors. To mitigate this issue, dimensionality reduction techniques (e.g., PCA, t-SNE) or feature selection methods can be applied before using K-NN.

*(An example to understand this concept better: imagine many people in a room, if we consider only 2 features (height and weight), we can find people that are similar to us. But if we consider 20 features (height, weight, age, income, education level, etc.), it becomes much harder to find people that are similar to us in all these dimensions, since the differences in one or more dimensions can make the overall distance between instances large. Quindi, all’aumentare delle dimensioni, tutti gli individui tendono ad essere ugualmente distanti tra loro, rendendo difficile distinguere chi è davvero simile o diverso.)*

### Recap - K-NN
- Accuracy: K-NN can achieve high accuracy, especially with a well-chosen k and appropriate distance metric. However, its performance can degrade in high-dimensional spaces due to the curse of dimensionality. If we have a good feature space, K-NN can perform very well.
- Interpretability: K-NN is not interpretable, as it is not really a model like decision trees and so does not provide a clear understanding of the decision-making process. The model's predictions are based on the local neighborhood of instances, making it difficult to explain why a specific prediction was made. The only way to explain a prediction is to show the k-nearest neighbors that influenced the decision.
- Incrementality: K-NN is inherently incremental, as new instances can be added to the training set without retraining the entire model. The model simply stores the new instances and uses them for future predictions. The only problem is that we must have the training set stored somewhere, so if we have a lot of data this could be a problem.
- Efficiency: Zero cost in training, of course. But K-NN can be computationally expensive while predicting, especially for large datasets, as it requires calculating distances to all training instances for each prediction. 
- Scalability: K-NN does not scale well in training set size, as the time complexity for making predictions is O(n), where n is the number of training instances. This can lead to slow prediction times for large datasets. Also, it does not scale well with high-dimensional feature spaces due to the curse of dimensionality.
- Robustness: Depends on the distance metric used and the choice of k. K-NN can be sensitive to noisy data and outliers, which can affect the accuracy of predictions. However, using a larger value of k or employing weighted K-NN can help mitigate this sensitivity. For the presence of missing values, K-NN can handle them by ignoring instances with missing values during distance calculations.

# Bayesian Classifiers (Naive Bayes)
Bayesian classifiers are a family of probabilistic classifiers based on Bayes' theorem, which provides a way to update the probability of a hypothesis (class label) given new evidence (feature values). 

Let C and X be random variables representing the class label and the feature vector, respectively. Bayes' theorem states that:  

$P(C|X) = \frac{P(X|C)P(C)}{P(X)}$  

With Bayesian classification:
- C is the class variable (the label we want to predict)
- X = <x1, x2, ..., xn> is the feature vector (the observed data)

Bayesian classification tells us how to **compute the posterior probability P(C|X) of each class C given the observed features X**.  
**The class with the highest posterior probability is then assigned to the instance** (sarebbe a dire, qual è la probabilità che l'istanza appartiene alla classe C dato che ha le caratteristiche X?).

In order to compute P(C|X), we need to estimate:
- The prior probability P(C) of each class C, which represents our initial belief about the class distribution before observing any features.  
So P(C) is simply the frequency of each class in the training set $P(C) = \frac{N_C}{N}$, where $N_C$ is the number of instances in class C and N is the total number of instances in the training set.
- The likelihood P(X|C) of the features X given each class C, which represents how likely the features are to occur for each class. In simpler terms, it tells us how probable it is to observe the features X if we know the instance belongs to class C.
- The evidence P(X), which is the overall probability of observing the features X across all classes. P(X) is constant for all C, so we can ignore it when comparing posterior probabilities for different classes.

But how to estimate P(X|C)? This is where the **Naive Bayes** classifier comes into play. Naive Bayes makes a simplifying assumption that the features in X are conditionally independent given the class C. This means that the presence or absence of one feature does not affect the presence or absence of another feature, given the class label. This assumption allows us to decompose the likelihood P(X|C) into the product of individual feature probabilities:

$P(X|C) = P(x_1, x_2, ..., x_n | C) = \prod_{i=1}^{n} P(x_i | C)$

Because of this strong assumption of independence among features, the Naive Bayes classifier is computationally efficient and easy to implement, but could lead to suboptimal performance if the independence assumption is violated in practice.

To compute P(x_i | C), we can use different approaches depending on the type of feature:
- For discrete features (categorical or ordinal), we can estimate P(x_i | C) using frequency counts from the training data. So:  
$P(x_k | C) = \frac{|x_{kC}|}{N_C}$
where $|x_{kC}|$ is the number of instances having value x_k for attribute k and belonging to class C, and N_C is the total number of instances in class C.  (semplicemente, quante volte compare il valore x_k nella classe C diviso il numero totale di istanze nella classe C, ricorda che X rappresenta il vettore delle caratteristiche di una nuova istanza da classificare, quindi x_k è il valore della caratteristica k per quella nuova istanza. |x_kC| rappresenta quante volte compare quel valore x_k nella classe C tra tutte le istanze nel training set)
- For continuous features, we can assume that the feature values follow a specific probability distribution (e.g., Gaussian distribution) within each class. We can then estimate the parameters of the distribution (e.g., mean and variance for Gaussian) from the training data and use them to compute P(x_i | C).  
We can also discretize continuous features into bins and treat them as categorical features, but this can lead to loss of information.

Below an example:

<img src="img_teoria/naive_bayes_1.png" width="50%">
<img src="img_teoria/naive_bayes_2.png" width="50%">
<img src="img_teoria/naive_bayes_3.png" width="50%">

Ossia dato un nuovo esempio X = ⟨rain, hot, high, false⟩ vogliamo decidere se appartiene alla classe P o N.
Per farlo dobbiamo calcolare P(P|X) e P(N|X) e scegliere la classe con la probabilità più alta.

### Recap - Bayesian Classifiers (Naive Bayes)
- Accuracy: Naive Bayes can achieve good accuracy, especially when the independence assumption holds true. However, its performance may degrade when features are highly correlated. Similar or lower than decision trees.
- Interpretability: Model and prediction are not interpretable, since it is based on probabilistic reasoning rather than explicit decision rules. Anyway, we can analyze the learned probabilities to understand the influence of different features on the classification.
- Incrementality: Naive Bayes is inherently incremental, as new instances can be added to the training set without retraining the entire model. The model simply updates the counts and probabilities based on the new instances.  
Also, differently from K-NN, we don't need to store all the training instances, but only the counts and probabilities, which makes it more efficient in terms of memory usage.
- Efficiency: Very fast to train and predict, as it involves simple probability calculations based on counts.
- Scalability: Scalable to large datasets and high-dimensional feature spaces, as the independence assumption allows for efficient computation of probabilities.
- Robustness: Not much robust: it is affected by noisy measures that affect the probability estimates, also we have to handle Nan values properly. It is extremely affected by attribute correlation.

# Support Vector Machines (SVM)
Support Vector Machines (SVM) are a powerful and versatile class of supervised learning algorithms used for classification and regression tasks. SVMs are particularly effective in high-dimensional spaces and are known for their ability to find optimal decision boundaries that maximize the margin between different classes.

We saw with decision trees how with every split we create axis-aligned decision boundaries between classes. SVMs instead create linear decision boundaries that are not necessarily axis-aligned, allowing them to capture more complex relationships between features and classes.

Let's see an example of binary classification with 2 features. In the figure, we see how the two classes could be separated in many ways, with different decision boundaries.  
The SVM algorithm aims to find the optimal hyperplane (in this case, a line) that separates the two classes (represented by circles and squares) with the **maximum margin**. The margin is defined as the *distance between the hyperplane and the nearest data points from each class, known as support vectors*.  
So in the example, between B1 and B2, the optimal hyperplane is B1, since it maximizes the margin between the two classes.

![](img_teoria/svm.png)

The idea of maximizing the margin is based on the intuition that a larger margin leads to better generalization performance on unseen data. By maximizing the distance between the decision boundary and the closest data points, SVMs aim to create a robust classifier that is less sensitive to noise and outliers in the training data.

If a problem is not linearly separable, we can use the **kernel trick** to transform the original feature space into a higher-dimensional space where a linear decision boundary can be found.
In the image below the boundary is not linear, but by applying a kernel function we project the data into a higher-dimensional space where a linear separator can be found.

<p align="center">
  <img src="img_teoria/svm_1.png" alt="SVM 1" width="48%"/>
  <img src="img_teoria/svm_2.png" alt="SVM 2" width="48%"/>
</p>

Of course this was the case for 2 features, but the same concept applies to higher dimensions as well. In that case, the decision boundary is a hyperplane that separates the classes in the higher-dimensional space.

### Recap - Support Vector Machines (SVM)
- **Accuracy**: SVMs are among the best classifiers in terms of accuracy because of their ability to find optimal decision boundaries and handle high-dimensional data effectively. (were among the best before deep learning and neural networks took over)
- **Interpretability**: SVMs are not very interpretable, as the decision boundary is defined by support vectors and kernel functions, which can be difficult to visualize and understand, especially in high-dimensional spaces. So SVM is mostly a black-box model.
- **Incrementality**: SVMs are not inherently incremental, since if new data points are added, the entire model needs to be retrained from scratch.
- **Efficiency**: SVMs can be computationally intensive, especially for large datasets, since they require solving an optimization problem. However, once trained, SVMs can make predictions relatively quickly.
- **Scalability**: SVMs scale linearly in the number of features, but their training time can grow quadratically or cubically (superlinear) with the number of instances, making them less suitable for very large datasets.
- **Robustness**: SVMs are generally robust to noisy data and outliers. For outliers we can use soft-margin SVMs, which allow some misclassifications to be tolerated in order to achieve a better overall decision boundary (i.e. we could permit some points to be inside the margin or even misclassified, in order to have a better generalization).